In [ ]:
!pip install openai datasets pandas python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.1 MB/s eta 0:00:00a 0:00:01
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.9/315.9 kB 8.9 MB/s eta 0:00:00
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [70]:
!pip install sympy antlr4-python3-runtime

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [71]:
import time
import json
import csv
import os
import pandas as pd
from dataclasses import dataclass, field, asdict
from typing import Optional
from openai import OpenAI  
from dotenv import load_dotenv
load_dotenv()

True

In [72]:
LLM_CONFIG = {
    "model": "gpt-4o-mini",
    "max_tokens": 1024,
    "temperature": 0.1,
}

COST_PER_1K_TOKENS = {
    "gpt-4o-mini":      {"input": 0.00015, "output": 0.0006},
    "qwen-2-72b":       {"input": 0.0009,  "output": 0.0009},
    "gemini-1.5-flash": {"input": 0.000075,"output": 0.0003},
    "llama-3.1-70b":    {"input": 0.00059, "output": 0.00079},
}

PATHS = {
    "gaia":       "../datasets/processed/processed_gaia.parquet",
    # "swe_bench":  "../datasets/processed/processed_swe_bench.parquet",
    # "math_hard":  "../datasets/processed/processed_math_hard.parquet",
    # "agentbench": "../datasets/processed/processed_agentbench.parquet",
}

In [73]:
RESULTS_DIR = "../datasets/baseline/"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [74]:
@dataclass
class Query:
    id: str
    dataset: str
    question: str
    ground_truth: str
    metadata: dict = field(default_factory=dict)

@dataclass
class Result:
    query_id: str
    dataset: str
    question: str
    ground_truth: str
    predicted: str
    is_correct: bool
    input_tokens: int
    output_tokens: int
    total_tokens: int
    cost_usd: float
    time_seconds: float
    model: str
    level: Optional[str] = None
    error: Optional[str] = None

In [75]:
def load_gaia(path: str) -> list[Query]:
    df = pd.read_parquet(path)

    print(f"[GAIA] Loaded {len(df)} rows")
    print(f"[GAIA] Columns: {df.columns.tolist()}")   # inspect once, then lock columns below

    queries = []
    for _, row in df.iterrows():
        queries.append(Query(
            id           = str(row.get("id",   row.name)),   # fallback to index
            dataset      = "GAIA",
            question     = str(row["query"]),                   # ← adjust col name if needed
            ground_truth = str(row["answer"]),               # ← adjust col name if needed
            metadata     = {
                "level": row.get("level", None),
                "file":  row.get("file_name", None),
            }
        ))
    return queries

In [76]:
load_gaia(PATHS["gaia"])

[GAIA] Loaded 165 rows
[GAIA] Columns: ['id', 'query', 'answer', 'level', 'annotator_steps', 'annotator_tools', 'file_name', 'steps_num', 'tool_num', 'time_taken']


[Query(id='c61d22de-5f6c-4958-a7f6-5e9707bd3466', dataset='GAIA', question='A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure with three axes, where each axis has a label word at both ends. Which of these words is used to describe a type of society in a Physics and Society article submitted to arXiv.org on August 11, 2016?', ground_truth='egalitarian', metadata={'level': '2', 'file': ''}),
 Query(id='17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc', dataset='GAIA', question='I’m researching species that became invasive after people who kept them as pets released them. There’s a certain species of fish that was popularized as a pet by being the main character of the movie Finding Nemo. According to the USGS, where was this fish found as a nonnative species, before the year 2020? I need the answer formatted as the five-digit zip codes of the places the species was found, separated by commas if there is more than one place.', ground_truth='34689', met

In [77]:
def load_unified_dataset() -> list[Query]:
    queries = []
    queries += load_gaia(PATHS["gaia"])

    # Uncomment as you add more processed datasets:
    # queries += load_swe_bench(PATHS["swe_bench"])
    # queries += load_math_hard(PATHS["math_hard"])
    # queries += load_agentbench(PATHS["agentbench"])

    print(f"\nTotal queries loaded: {len(queries)}")
    return queries

In [78]:
load_unified_dataset()

[GAIA] Loaded 165 rows
[GAIA] Columns: ['id', 'query', 'answer', 'level', 'annotator_steps', 'annotator_tools', 'file_name', 'steps_num', 'tool_num', 'time_taken']

Total queries loaded: 165


[Query(id='c61d22de-5f6c-4958-a7f6-5e9707bd3466', dataset='GAIA', question='A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure with three axes, where each axis has a label word at both ends. Which of these words is used to describe a type of society in a Physics and Society article submitted to arXiv.org on August 11, 2016?', ground_truth='egalitarian', metadata={'level': '2', 'file': ''}),
 Query(id='17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc', dataset='GAIA', question='I’m researching species that became invasive after people who kept them as pets released them. There’s a certain species of fish that was popularized as a pet by being the main character of the movie Finding Nemo. According to the USGS, where was this fish found as a nonnative species, before the year 2020? I need the answer formatted as the five-digit zip codes of the places the species was found, separated by commas if there is more than one place.', ground_truth='34689', met

In [79]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

SYSTEM_PROMPT = """You are an expert question answering system evaluated on the GAIA benchmark.

## Output Rules
- Return ONLY the final answer — nothing else
- Answers are always one of: a single word, a number, a short phrase, or a name
- No explanations, no sentences, no punctuation at the end
- No preamble like "The answer is..." or "Based on..."
- If the answer is a number, return just a single number after one comma dont return (e.g. 42, 3.14)
- If the answer is a name, return just the name (e.g. Paris, Einstein)
- If the answer is a word, return just that word (e.g. egalitarian, blue)
- If the answer is a short phrase, return just that phrase (e.g. "New York", "machine learning")

## Now answer the following question with ONLY the final answer:
"""
def call_direct_llm(query: Query) -> Result:
    model   = LLM_CONFIG["model"]
    start   = time.perf_counter()
    error   = None
    predicted = ""
    usage   = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

    try:
        response = client.chat.completions.create(
            model=model,
            max_tokens=LLM_CONFIG["max_tokens"],
            temperature=LLM_CONFIG["temperature"],
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": query.question},
            ],
        )
        predicted = response.choices[0].message.content.strip()
        usage = {
            "prompt_tokens":     response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens":      response.usage.total_tokens,
        }
    except Exception as e:
        error = str(e)
        error_type = type(e).__name__

    elapsed = time.perf_counter() - start

    rates = COST_PER_1K_TOKENS.get(model, {"input": 0, "output": 0})
    cost  = (
        usage["prompt_tokens"]     / 1000 * rates["input"] +
        usage["completion_tokens"] / 1000 * rates["output"]
    )

    is_correct = predicted.strip().lower() == query.ground_truth.strip().lower()

    return Result(
        query_id     = query.id,
        dataset      = query.dataset,
        question     = query.question,
        ground_truth = query.ground_truth,
        predicted    = predicted,
        is_correct   = is_correct,
        input_tokens = usage["prompt_tokens"],
        output_tokens= usage["completion_tokens"],
        total_tokens = usage["total_tokens"],
        cost_usd     = round(cost, 6),
        time_seconds = round(elapsed, 3),
        model        = model,
        level        = query.metadata.get("level"),
        error        = error,
    )

In [80]:
def run_baseline(queries: list[Query]) -> list[Result]:
    results = []
    for i, q in enumerate(queries):
        print(f"[{i+1}/{len(queries)}] {q.dataset} | {q.id}")
        print(f"   ❓ Question : {q.question[:100]}...")   # trim long questions
        print(f"   ✅ Expected : {q.ground_truth}")

        r = call_direct_llm(q)
        results.append(r)

        status = "✓" if r.is_correct else "✗"
        print(f"   🤖 Predicted: {r.predicted}")
        print(f"   {status} correct={r.is_correct}  tokens={r.total_tokens}  "
              f"cost=${r.cost_usd:.5f}  time={r.time_seconds}s")
        if r.error:
            print(f"   ⚠ error: {r.error} ({r.error_type})")
        print("-" * 60)

    return results

In [81]:
queries = load_unified_dataset()
results = run_baseline(queries)

[GAIA] Loaded 165 rows
[GAIA] Columns: ['id', 'query', 'answer', 'level', 'annotator_steps', 'annotator_tools', 'file_name', 'steps_num', 'tool_num', 'time_taken']

Total queries loaded: 165
[1/165] GAIA | c61d22de-5f6c-4958-a7f6-5e9707bd3466
   ❓ Question : A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure w...
   ✅ Expected : egalitarian
   🤖 Predicted: egalitarian
   ✓ correct=True  tokens=269  cost=$0.00004  time=1.448s
------------------------------------------------------------
[2/165] GAIA | 17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc
   ❓ Question : I’m researching species that became invasive after people who kept them as pets released them. There...
   ✅ Expected : 34689
   🤖 Predicted: 33139, 33140, 33141, 33142, 33143
   ✗ correct=False  tokens=311  cost=$0.00006  time=0.987s
------------------------------------------------------------
[3/165] GAIA | 04a04a9b-226c-43fd-b319-d5e89743676f
   ❓ Question : If we assume all articles publ

KeyboardInterrupt: 

In [149]:

def evaluate(results: list[Result]):
    total   = len(results)
    correct = sum(r.is_correct for r in results)

    print("\n─── BASELINE SUMMARY ───────────────────────────")
    print(f"  Model         : {results[0].model}")
    print(f"  Total queries : {total}")
    print(f"  Accuracy      : {correct}/{total} = {correct/total:.1%}")
    print(f"  Total tokens  : {sum(r.total_tokens  for r in results):,}")
    print(f"  Total cost    : ${sum(r.cost_usd     for r in results):.4f}")
    print(f"  Avg latency   : {sum(r.time_seconds  for r in results)/total:.2f}s")
    print(f"  Errors        : {sum(1 for r in results if r.error)}")

    # Per-dataset breakdown
    print("\n─── PER DATASET ─────────────────────────────────")
    for ds in sorted({r.dataset for r in results}):
        ds_res = [r for r in results if r.dataset == ds]
        acc = sum(r.is_correct for r in ds_res) / len(ds_res)
        print(f"  [{ds}]  accuracy={acc:.1%}  n={len(ds_res)}"
              f"  cost=${sum(r.cost_usd for r in ds_res):.4f}")

    # GAIA level breakdown (if metadata present)
    gaia_res = [r for r in results if r.dataset == "GAIA"]
    if gaia_res:
        df_eval = pd.DataFrame([asdict(r) for r in gaia_res])
        if "level" in df_eval.columns:
            print("\n─── GAIA BY LEVEL ───────────────────────────────")
            # print(df_eval.groupby("level")["is_correct"].mean().to_string())
            print(
            (df_eval.groupby("level")["is_correct"].mean() * 100)
            .round(2)
            .astype(str) + "%"
)



In [150]:
evaluate(results)


─── BASELINE SUMMARY ───────────────────────────
  Model         : gpt-4o-mini
  Total queries : 165
  Accuracy      : 9/165 = 5.5%
  Total tokens  : 44,315
  Total cost    : $0.0069
  Avg latency   : 0.74s
  Errors        : 0

─── PER DATASET ─────────────────────────────────
  [GAIA]  accuracy=5.5%  n=165  cost=$0.0069

─── GAIA BY LEVEL ───────────────────────────────
level
1    5.66%
2    6.98%
3     0.0%
Name: is_correct, dtype: str


In [151]:
def save_results(results: list[Result]):
    model_tag = LLM_CONFIG["model"].replace("/", "-")
    
    # Save path: datasets/baseline/gaia/v1_{model_name}.parquet
    save_dir = os.path.join("..", "datasets", "baseline", "gaia")
    os.makedirs(save_dir, exist_ok=True)
    
    parquet_path = os.path.join(save_dir, f"v1_{model_tag}.parquet")
    
    pd.DataFrame([asdict(r) for r in results]).to_parquet(parquet_path, index=False)
    
    print(f"\n✅ Saved → {parquet_path}")
    print(f"   Rows    : {len(results)}")
    print(f"   Model   : {model_tag}")
    print(f"   Correct : {sum(r.is_correct for r in results)}/{len(results)}")

In [152]:
save_results(results)


✅ Saved → ../datasets/baseline/gaia/v1_gpt-4o-mini.parquet
   Rows    : 165
   Model   : gpt-4o-mini
   Correct : 9/165


### CoT

In [82]:
import time
import os
import re
import pandas as pd
from dataclasses import dataclass, field, asdict
from typing import Optional
from openai import OpenAI

# ─────────────────────────────────────────────────────────────
# Wei et al. 2022 — Core Principles Implemented:
#   1. Few-shot exemplars (8 per task) with hand-crafted chains
#   2. Format: Q → reasoning steps → "The answer is X"
#   3. Dataset-specific exemplar sets (arithmetic / commonsense)
#   4. No model fine-tuning — pure prompting only
#   5. Zero-shot fallback: "Let's think step by step"
# ─────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────
LLM_CONFIG = {
    "model":       "gpt-4o-mini",
    "max_tokens":  2048,      # CoT needs more tokens for reasoning
    "temperature": 0.1,       # deterministic — paper used greedy decoding
}

COST_PER_1K_TOKENS = {
    "gpt-4o-mini":      {"input": 0.00015, "output": 0.0006},
    # "qwen-2-72b":       {"input": 0.0009,  "output": 0.0009},
    # "gemini-1.5-flash": {"input": 0.000075,"output": 0.0003},
    # "llama-3.1-70b":    {"input": 0.00059, "output": 0.00079},
}

PATHS = {
    "gaia":      "../datasets/processed/processed_gaia.parquet",
    # "math_hard": "../datasets/processed/processed_math_hard.parquet",
    "mmlu_pro":  "../datasets/processed/processed_mmlu_pro.parquet",
}

In [83]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [84]:
@dataclass
class Query:
    id: str
    dataset: str          # GAIA | MATH-HARD | MMLU-PRO
    question: str
    ground_truth: str
    level: Optional[str]  = None
    options: Optional[str]= None   # MMLU-Pro has MCQ options
    subject: Optional[str]= None   # MMLU-Pro subject
    metadata: dict        = field(default_factory=dict)

@dataclass
class Result:
    query_id: str
    dataset: str
    question: str
    ground_truth: str
    full_reasoning: str   # complete CoT chain
    predicted: str        # extracted final answer only
    is_correct: bool
    input_tokens: int
    output_tokens: int
    total_tokens: int
    cost_usd: float
    time_seconds: float
    model: str
    cot_type: str         # "few_shot" | "zero_shot"
    level: Optional[str]  = None
    subject: Optional[str]= None
    error: Optional[str]  = None


In [85]:
GAIA_EXEMPLARS = """Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
A: Roger started with 5 balls. 2 cans × 3 balls = 6 balls. 5 + 6 = 11. The answer is 11.

Q: A store sells apples at $1.50 each. A 10% discount is applied if you buy more than 4. How much do 6 apples cost?
A: Cost without discount = 6 × $1.50 = $9.00. Discount = 10% of $9.00 = $0.90. Final cost = $9.00 − $0.90 = $8.10. The answer is 8.10.

Q: A paper was submitted to arXiv in June 2022. It discusses AI regulation and includes a figure with axes labeled with opposing terms. What type of methodology involves structured oversight of automated decision systems?
A: AI regulation papers typically discuss governance frameworks. Structured oversight of automated systems is referred to as algorithmic accountability. The answer is algorithmic accountability.

Q: A train travels from City A to City B at 60 km/h and returns at 40 km/h. What is the average speed for the whole trip?
A: Average speed for round trip = 2 × (s1 × s2) / (s1 + s2). = 2 × (60 × 40) / (60 + 40) = 4800 / 100 = 48 km/h. The answer is 48.

Q: Which country won the most gold medals in the 2020 Summer Olympics?
A: The 2020 Summer Olympics were held in Tokyo in 2021. The United States topped the medal table with 39 gold medals. The answer is United States.

Q: A rectangle has a perimeter of 36 cm and a width of 8 cm. What is its area?
A: Perimeter = 2 × (length + width). 36 = 2 × (length + 8). length + 8 = 18. length = 10. Area = 10 × 8 = 80 cm². The answer is 80.

Q: If a species was introduced as a pet and later became invasive, what government agency in the US tracks its nonnative sightings?
A: In the United States, the U.S. Geological Survey (USGS) maintains the Nonindigenous Aquatic Species (NAS) database which tracks nonnative species sightings. The answer is USGS.

Q: What is the last letter of the last word in the phrase "chain of thought prompting"?
A: The last word is "prompting". The last letter of "prompting" is "g". The answer is g."""




# ── MMLU-PRO — multiple choice, domain knowledge ──
MMLU_EXEMPLARS = """Q: Which of the following best describes the process of meiosis?
Options: A) Cell growth and protein synthesis  B) DNA replication without cell division  C) Cell division that produces gametes with half the chromosome number  D) Mitotic division of somatic cells
A: Meiosis is a specialized cell division that produces gametes (sperm and eggs). It results in four cells each with half the original chromosome number (haploid). This distinguishes it from mitosis which produces identical diploid cells. The answer is C.

Q: What is the time complexity of binary search on a sorted array of n elements?
Options: A) O(n)  B) O(log n)  C) O(n log n)  D) O(n²)
A: Binary search works by repeatedly halving the search space. Starting with n elements, after 1 step: n/2, after 2 steps: n/4, after k steps: n/2^k = 1, so k = log₂(n). The answer is B.

Q: In economics, what does the term "opportunity cost" refer to?
Options: A) The direct monetary cost of a purchase  B) The value of the next best alternative foregone  C) The total cost of production  D) The marginal cost of one additional unit
A: Opportunity cost is a core economic concept referring to what you give up by making a choice — specifically the value of the best alternative not chosen. It is not the direct monetary cost but the implicit cost of foregone options. The answer is B.

Q: Which law states that the pressure of a gas is inversely proportional to its volume at constant temperature?
Options: A) Charles's Law  B) Avogadro's Law  C) Boyle's Law  D) Gay-Lussac's Law
A: Boyle's Law states P ∝ 1/V at constant temperature, i.e., P₁V₁ = P₂V₂. Charles's Law relates volume and temperature. Gay-Lussac's Law relates pressure and temperature. The answer is C.

Q: Who wrote the philosophical work "Critique of Pure Reason"?
Options: A) David Hume  B) René Descartes  C) John Locke  D) Immanuel Kant
A: "Critique of Pure Reason" (Kritik der reinen Vernunft) was published in 1781 by Immanuel Kant. It is a foundational text in modern philosophy examining the nature and limits of human knowledge. The answer is D.

Q: In Python, what does the "yield" keyword do?
Options: A) Terminates a function immediately  B) Returns a value and pauses the function, making it a generator  C) Imports an external module  D) Declares a global variable
A: "yield" turns a function into a generator. When called, it returns a value to the caller and suspends the function's state, allowing it to resume from that point on the next call. The answer is B.

Q: What is the powerhouse of the cell?
Options: A) Nucleus  B) Ribosome  C) Mitochondria  D) Golgi apparatus
A: Mitochondria are responsible for producing ATP through cellular respiration, providing energy for cell functions. The nucleus stores DNA, ribosomes synthesize proteins, Golgi processes and packages proteins. The answer is C.

Q: Which of the following is NOT a property of a normal distribution?
Options: A) It is symmetric about the mean  B) Mean equals median equals mode  C) It has heavy tails compared to a uniform distribution  D) About 68% of data falls within 1 standard deviation of the mean
A: A normal distribution is symmetric, has mean=median=mode, and follows the 68-95-99.7 rule. Normal distributions actually have lighter tails than heavy-tailed distributions like Cauchy. Option C incorrectly claims it has heavy tails — this is NOT a property. The answer is C."""




In [86]:
# Map dataset → exemplar block
EXEMPLARS = {
    "GAIA":     GAIA_EXEMPLARS,
    # "MATH-HARD": MATH_EXEMPLARS,
    "MMLU-PRO": MMLU_EXEMPLARS,
}

In [ ]:
# ─────────────────────────────────────────────
# 4. BUILD PROMPTS (Wei et al. style)
# ─────────────────────────────────────────────
def build_few_shot_prompt(query: Query) -> list[dict]:
    """
    Wei et al. 2022 Few-Shot CoT:
    System sets the task context.
    User message = 8 exemplars + new question.
    """
    exemplars = EXEMPLARS.get(query.dataset, GAIA_EXEMPLARS)

    system = ( """You are an expert QA system using step-by-step logical reasoning.

Rules:
1. Understand the question fully.
2. Solve using clear, logical steps , optimally.
3. Apply correct formulas and facts.
4. Ensure accuracy and consistency.
5. Do not guess—reason carefully.

Output (STRICT):
- End EXACTLY with: The answer is <final_answer>.
- No text after this line.
- Final answer must be one of:
  • single number
  • option letter (A–J)
  • one word
  • short phrase
- No explanation in the final answer line.
""")
#    

    # For MMLU-Pro, append options to the question
    # question_text = query.question
    # if query.dataset == "MMLU-PRO" and query.options:
    #     question_text = f"{query.question}\nOptions: {query.options}"
    
    question_text = query.question

    if query.dataset == "MMLU-PRO" and query.options is not None and len(query.options) > 0:
        options_str = "\n".join(
            [f"{chr(65+i)}. {opt}" for i, opt in enumerate(query.options)]
        )
        question_text = f"{query.question}\n{options_str}"

    user = f"{exemplars}\n\nQ: {question_text}\nA:"

    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]


In [176]:
def build_zero_shot_prompt(query: Query) -> list[dict]:
    """
    Kojima et al. 2022 Zero-Shot CoT:
    Append 'Let's think step by step' — used as fallback.
    """
    question_text = query.question
    if query.dataset == "MMLU-PRO" and query.options:
        question_text = f"{query.question}\nOptions: {query.options}"

    return [
        {"role": "system", "content": "You are an expert reasoning system."},
        {"role": "user",   "content": f"Q: {question_text}\nA: Let's think step by step."},
    ]


In [177]:
def extract_answer(response_text: str, dataset: str) -> tuple[str, str]:
    """
    Returns (full_reasoning, extracted_answer)
    Extracts from "The answer is X" pattern — Wei et al. 2022 standard.
    """
    full_reasoning = response_text.strip()

    # Primary: "The answer is X." pattern (Wei et al. standard)
    match = re.search(
        r"[Tt]he answer is[:\s]+([^\n.]+)",
        response_text
    )
    if match:
        answer = match.group(1).strip().rstrip(".")
        return full_reasoning, answer

    # MMLU-Pro fallback: look for single letter A/B/C/D/E at end
    if dataset == "MMLU-PRO":
        match = re.search(r"\b([A-E])\b\.?\s*$", response_text.strip())
        if match:
            return full_reasoning, match.group(1)

    # Last line fallback
    lines  = [l.strip() for l in response_text.strip().split("\n") if l.strip()]
    answer = lines[-1].rstrip(".") if lines else response_text
    return full_reasoning, answer


In [178]:
def normalize(text: str) -> str:
    return text.strip().lower().rstrip(".,:;!")


In [179]:
def load_gaia(path: str) -> list[Query]:
    df = pd.read_parquet(path)
    print(f"[GAIA]      Loaded {len(df)} rows | Columns: {df.columns.tolist()}")
    return [
        Query(
            id           = str(row.get("id", i)),
            dataset      = "GAIA",
            question     = str(row["query"]),
            ground_truth = str(row["answer"]),
            level        = str(row.get("level", "")),
            metadata     = {"file": row.get("file_name"), "steps": row.get("steps_num")}
        )
        for i, (_, row) in enumerate(df.iterrows())
    ]

In [180]:
def load_mmlu_pro(path: str) -> list[Query]:
    df = pd.read_parquet(path)
    print(f"[MMLU-PRO]  Loaded {len(df)} rows | Columns: {df.columns.tolist()}")
    queries = []
    for i, (_, row) in enumerate(df.iterrows()):
        # Build options string from columns (adjust col names to your parquet)
        options = row.get("options", None)
        if isinstance(options, list):
            labels  = ["A", "B", "C", "D", "E", "F", "G", "H"]
            options = "  ".join(f"{labels[j]}) {opt}" for j, opt in enumerate(options))

        queries.append(Query(
            id           = str(row.get("id", i)),
            dataset      = "MMLU-PRO",
            question     = str(row.get("query", "")),
            ground_truth = str(row.get("answer", row.get("answer_index", ""))),
            level        = str(row.get("category", row.get("subject", ""))),
            options      = options,
            subject      = str(row.get("category", row.get("subject", ""))),
        ))
    return queries


In [181]:


def load_all() -> list[Query]:
    queries = []
    queries += load_gaia(PATHS["gaia"])
    # queries += load_math_hard(PATHS["math_hard"]) 
    queries += load_mmlu_pro(PATHS["mmlu_pro"][:200])
    print(f"\nTotal: {len(queries)} queries across 3 datasets\n")
    return queries

# ─────────────────────────────────────────────
# 7. CoT LLM CALL — Wei et al. 2022
# ─────────────────────────────────────────────
def call_cot_llm(query: Query, cot_type: str = "few_shot") -> Result:
    model   = LLM_CONFIG["model"]
    start   = time.perf_counter()
    error   = None
    full_reasoning = ""
    predicted      = ""
    usage   = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

    messages = (
        build_few_shot_prompt(query)
        if cot_type == "few_shot"
        else build_zero_shot_prompt(query)
    )

    try:
        response = client.chat.completions.create(
            model       = model,
            max_tokens  = LLM_CONFIG["max_tokens"],
            temperature = LLM_CONFIG["temperature"],   # greedy — Wei et al. 2022
            messages    = messages,
        )
        raw_text = response.choices[0].message.content.strip()
        full_reasoning, predicted = extract_answer(raw_text, query.dataset)

        usage = {
            "prompt_tokens":     response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens":      response.usage.total_tokens,
        }
    except Exception as e:
        error = str(e)

    elapsed  = time.perf_counter() - start
    rates    = COST_PER_1K_TOKENS.get(model, {"input": 0, "output": 0})
    cost     = (
        usage["prompt_tokens"]     / 1000 * rates["input"] +
        usage["completion_tokens"] / 1000 * rates["output"]
    )
    # is_correct = normalize(predicted) == normalize(query.ground_truth)
    is_correct = compute_correctness(predicted, query.ground_truth, query.dataset)


    return Result(
        query_id       = query.id,
        dataset        = query.dataset,
        question       = query.question,
        ground_truth   = query.ground_truth,
        full_reasoning = full_reasoning,
        predicted      = predicted,
        is_correct     = is_correct,
        input_tokens   = usage["prompt_tokens"],
        output_tokens  = usage["completion_tokens"],
        total_tokens   = usage["total_tokens"],
        cost_usd       = round(cost, 6),
        time_seconds   = round(elapsed, 3),
        model          = model,
        cot_type       = cot_type,
        level          = query.level,
        subject        = query.subject,
        error          = error,
    )

In [182]:
def run_cot_experiment(
    queries:  list[Query],
    cot_type: str = "few_shot"   # "few_shot" | "zero_shot"
) -> list[Result]:

    print(f"\n{'='*60}")
    print(f"  Wei et al. 2022 CoT — {cot_type.upper()}")
    print(f"  Model  : {LLM_CONFIG['model']}")
    print(f"  Queries: {len(queries)}")
    print(f"{'='*60}\n")

    results = []
    for i, q in enumerate(queries):
        print(f"[{i+1}/{len(queries)}] {q.dataset} | Level: {q.level} | {q.id}")
        print(f"   ❓ Q  : {q.question[:100]}...")
        print(f"   ✅ GT : {q.ground_truth}")

        r = call_cot_llm(q, cot_type=cot_type)
        results.append(r)

        status = "✓" if r.is_correct else "✗"
        print(f"   🧠 CoT: {r.full_reasoning[:120].strip()}...")
        print(f"   🤖 Ans: {r.predicted}")
        print(f"   {status}  tokens={r.total_tokens}  cost=${r.cost_usd:.5f}  "
              f"time={r.time_seconds}s")
        if r.error:
            print(f"   ⚠ {r.error}")
        print("-" * 65)

    return results

In [183]:
import re
import unicodedata
import pandas as pd
from dataclasses import asdict
from typing import Optional
 
# ─────────────────────────────────────────────────────────────
# CORRECTED EVALUATION BLOCK — Wei et al. 2022 CoT
# Handles all 3 datasets with dataset-specific logic:
#   • GAIA     → fuzzy / substring / numeric matching
#   • MATH-HARD → sympy symbolic + numeric fallback
#   • MMLU-PRO  → letter-only extraction + match
# ─────────────────────────────────────────────────────────────
 
# ── optional but highly recommended ──────────────────────────
try:
    from sympy import sympify, simplify, N
    from sympy.parsing.latex import parse_latex
    SYMPY_AVAILABLE = True
except ImportError:
    SYMPY_AVAILABLE = False
    print("[WARN] sympy not installed — MATH-HARD will use string fallback only.")
    print("       Run: pip install sympy antlr4-python3-runtime")
 
 
# ═════════════════════════════════════════════════════════════
# SECTION 1 — ANSWER EXTRACTION (per dataset)
# ═════════════════════════════════════════════════════════════
 
def extract_answer(response_text: str, dataset: str) -> tuple[str, str]:
    """
    Returns (full_reasoning, extracted_answer).
 
    Extraction strategy varies per dataset:
      GAIA      → "The answer is X" → free-form string
      MATH-HARD → "The answer is X" → preserve math expressions
      MMLU-PRO  → "The answer is X" → grab only the leading letter A–J
    """
    full_reasoning = response_text.strip()
 
    # ── Step 1: Try canonical Wei et al. pattern ─────────────
    match = re.search(
        r"[Tt]he answer is[:\s]+([^\n]+)",
        response_text
    )
 
    if match:
        raw_answer = match.group(1).strip().rstrip(".")
 
        if dataset == "MMLU-PRO":
            # Extract ONLY the letter — ignore trailing explanation
            # Handles: "C", "C.", "C)", "C) Mitochondria", "(C)"
            letter_match = re.match(r"\(?([A-Ja-j])\)?[\.\):\s]?", raw_answer)
            answer = letter_match.group(1).upper() if letter_match else raw_answer
 
        elif dataset == "MATH-HARD":
            # Normalise unicode → ASCII, strip spaces for consistent comparison
            answer = re.sub(r"\s+", "",
                raw_answer
                .replace("\u2212", "-")
                .replace("\u00d7", "*")
                .replace("\u00f7", "/")
                .replace("π",      "pi")
                .replace("\u03c0", "pi")
                .replace("²",      "**2")
                .replace("³",      "**3")
                .replace("^",      "**")
            ).strip()
        else:
            # GAIA — plain string, light cleanup
            answer = raw_answer
 
        return full_reasoning, answer
 
    # ── Step 2: Dataset-specific fallbacks ───────────────────
 
    if dataset == "MMLU-PRO":
        # Scan whole response for the last standalone A–J letter
        letter_matches = re.findall(r"\b([A-Ja-j])\b", response_text)
        if letter_matches:
            return full_reasoning, letter_matches[-1].upper()
 
    if dataset == "MATH-HARD":
        # Look for boxed LaTeX: \boxed{answer}
        boxed = re.search(r"\\boxed\{([^}]+)\}", response_text)
        if boxed:
            return full_reasoning, boxed.group(1).strip()
 
        # Look for "= <value>" at end of a line (common math pattern)
        eq_match = re.search(r"=\s*([^\n=]+)\s*$", response_text, re.MULTILINE)
        if eq_match:
            return full_reasoning, eq_match.group(1).strip()
 
    # ── Step 3: Universal last-line fallback ─────────────────
    lines = [l.strip() for l in response_text.strip().split("\n") if l.strip()]
    answer = lines[-1].rstrip(".") if lines else response_text.strip()
    return full_reasoning, answer
 
 
# ═════════════════════════════════════════════════════════════
# SECTION 2 — NORMALISATION HELPERS
# ═════════════════════════════════════════════════════════════
 
def normalize_string(text: str) -> str:
    """
    General string normalisation for GAIA / fallback use:
      - lowercase
      - strip punctuation/whitespace
      - collapse unicode (é → e)
      - normalise common abbreviations
    """
    text = text.strip().lower()
    # Unicode normalise: accented → ASCII where possible
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    # Strip trailing punctuation
    text = text.rstrip(".,:;!")
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text)
    return text
 
 
# Map of known equivalent forms — extend as needed
GAIA_ALIASES = {
    "united states":    ["us", "usa", "u.s.", "u.s.a.", "united states of america"],
    "united kingdom":   ["uk", "u.k.", "britain", "great britain"],
    "world war ii":     ["wwii", "ww2", "world war 2", "second world war"],
    "artificial intelligence": ["ai"],
    "united nations":   ["un", "u.n."],
}
 
def _build_alias_map() -> dict[str, str]:
    """Returns alias → canonical form lookup."""
    alias_map = {}
    for canonical, aliases in GAIA_ALIASES.items():
        for alias in aliases:
            alias_map[normalize_string(alias)] = normalize_string(canonical)
    return alias_map
 
_ALIAS_MAP = _build_alias_map()
 
 
def resolve_alias(text: str) -> str:
    """Replace known alias with canonical form."""
    norm = normalize_string(text)
    return _ALIAS_MAP.get(norm, norm)
 
 
def try_parse_number(text: str) -> Optional[float]:
    """Try to parse text as a float, handling commas and $ signs."""
    text = text.replace(",", "").replace("$", "").replace("%", "").strip()
    try:
        return float(text)
    except ValueError:
        return None
 
 
# ═════════════════════════════════════════════════════════════
# SECTION 3 — DATASET-SPECIFIC EVALUATORS
# ═════════════════════════════════════════════════════════════
 
# ── 3a. GAIA Evaluator ───────────────────────────────────────
 
def eval_gaia(predicted: str, ground_truth: str) -> bool:
    """
    GAIA answers are free-form: names, numbers, acronyms, dates.
    Strategy (ordered by strictness):
      1. Exact match after normalisation
      2. Alias resolution (US == United States)
      3. Numeric match (handles units/formatting differences)
      4. Substring containment (ground truth ⊆ predicted or vice versa)
    """
    p_norm = normalize_string(predicted)
    g_norm = normalize_string(ground_truth)
 
    # 1. Exact
    if p_norm == g_norm:
        return True
 
    # 2. Alias resolution
    if resolve_alias(p_norm) == resolve_alias(g_norm):
        return True
 
    # 3. Numeric match — both parseable as numbers
    p_num = try_parse_number(predicted)
    g_num = try_parse_number(ground_truth)
    if p_num is not None and g_num is not None:
        return abs(p_num - g_num) < 1e-6
 
    # 4. Substring containment (handles extra words in answer)
    if g_norm and (g_norm in p_norm or p_norm in g_norm):
        return True
 
    return False
 
 
# ── 3b. MATH-HARD Evaluator ──────────────────────────────────
 
def _preprocess_math_expr(expr: str) -> str:
    """
    Pre-process a math expression string before passing to sympy.
    Converts common unicode/notation variants to sympy-parseable ASCII.
    """
    result = (
        expr
        .replace("π", "pi")        # greek pi → sympy pi
        .replace("\u03c0", "pi")   # unicode pi
        .replace("\u2212", "-")    # unicode minus → ASCII minus
        .replace("\u00d7", "*")    # × → *
        .replace("\u00f7", "/")    # ÷ → /
        .replace("²", "**2")       # superscript 2 → power
        .replace("³", "**3")       # superscript 3 → power
        .replace("^", "**")        # caret → power
        .strip()
    )
    # Insert implicit multiplication: "14pi" → "14*pi", "2x" → "2*x"
    result = re.sub(r"(\d)(pi\b|[a-df-wyzA-Z])", r"\1*\2", result)
    # Remove spaces for normalised string comparison
    result = re.sub(r"\s+", "", result)
    return result
 
 
def _sympy_equal(expr1: str, expr2: str) -> Optional[bool]:
    """
    Try to evaluate symbolic equality using sympy.
    Returns True/False/None (None = could not parse).
    """
    if not SYMPY_AVAILABLE:
        return None
 
    e1 = _preprocess_math_expr(expr1)
    e2 = _preprocess_math_expr(expr2)
 
    # Try plain sympify first, then LaTeX parsing
    for parse_fn in [sympify, parse_latex]:
        try:
            a = parse_fn(e1)
            b = parse_fn(e2)
            diff = simplify(a - b)
            if diff == 0:
                return True
            # Numeric check as secondary (handles irrational expressions like 14π vs 43.98)
            if abs(complex(N(diff))) < 1e-2:
                return True
        except Exception:
            continue
    return None
 
 
def _normalise_math_string(text: str) -> str:
    """
    Normalise a math expression string for string-level comparison.
    Strips spaces, lowercases, normalises operators.
    """
    text = text.lower().strip()
    text = re.sub(r"\s+", "", text)        # remove all whitespace
    text = text.replace("^", "**")         # unify exponent notation
    text = text.replace("×", "*")
    text = text.replace("÷", "/")
    text = text.replace("π", "pi")
    text = text.rstrip(".")
    return text
 
 
def eval_math(predicted: str, ground_truth: str) -> bool:
    """
    MATH-HARD answers can be numbers, expressions, or equations.
    Strategy:
      1. Sympy symbolic equality (exact + numeric)
      2. Normalised string match
      3. Float comparison fallback
    """
    # 1. Sympy
    sympy_result = _sympy_equal(predicted, ground_truth)
    if sympy_result is not None:
        return sympy_result
 
    # 2. Normalised string
    p = _normalise_math_string(predicted)
    g = _normalise_math_string(ground_truth)
    if p == g:
        return True
 
    # 3. Float fallback
    p_num = try_parse_number(predicted)
    g_num = try_parse_number(ground_truth)
    if p_num is not None and g_num is not None:
        return abs(p_num - g_num) < 1e-6
 
    return False
 
 
# ── 3c. MMLU-PRO Evaluator ───────────────────────────────────
 
def eval_mmlu(predicted: str, ground_truth: str) -> bool:
    """
    MMLU-PRO: single letter answer A–J.
    Extracts only the letter from both sides before comparing.
    Handles: "C", "(C)", "C.", "C) explanation", "answer: C"
    """
    def extract_letter(text: str) -> str:
        text = text.strip()
        match = re.search(r"\b([A-Ja-j])\b", text)
        if match:
            return match.group(1).upper()
        # Fallback: first character if it's a letter
        if text and text[0].upper() in "ABCDEFGHIJ":
            return text[0].upper()
        return text.upper()
 
    return extract_letter(predicted) == extract_letter(ground_truth)
 
 
# ═════════════════════════════════════════════════════════════
# SECTION 4 — UNIFIED ROUTER
# ═════════════════════════════════════════════════════════════
 
def compute_correctness(predicted: str, ground_truth: str, dataset: str) -> bool:
    """
    Single entry point — routes to the correct evaluator per dataset.
    Use this in call_cot_llm() instead of the raw normalize() comparison.
 
    Example:
        is_correct = compute_correctness(predicted, query.ground_truth, query.dataset)
    """
    if not predicted or not ground_truth:
        return False
 
    if dataset == "MMLU-PRO":
        return eval_mmlu(predicted, ground_truth)
    elif dataset == "MATH-HARD":
        return eval_math(predicted, ground_truth)
    else:  # GAIA + any unknown dataset
        return eval_gaia(predicted, ground_truth)
 
 
# ═════════════════════════════════════════════════════════════
# SECTION 5 — ENRICHED EVALUATE() FUNCTION
# Drop-in replacement for the original evaluate() in the main file
# ═════════════════════════════════════════════════════════════
 
def evaluate(results: list) -> pd.DataFrame:
    """
    Enriched evaluation with:
      - Per-dataset accuracy breakdown
      - Level/subject sub-breakdown
      - Error analysis
      - Correctness re-checked using dataset-specific evaluators
        (re-evaluation is non-destructive — stored in 'is_correct_strict')
    """
    df = pd.DataFrame([asdict(r) for r in results])
 
    # ── Re-evaluate with correct per-dataset logic ────────────
    # (Adds 'is_correct_strict' — original 'is_correct' untouched for comparison)
    df["is_correct_strict"] = df.apply(
        lambda row: compute_correctness(
            str(row["predicted"]),
            str(row["ground_truth"]),
            row["dataset"]
        ),
        axis=1
    )
 
    model    = results[0].model
    cot_type = results[0].cot_type
 
    print(f"\n{'='*65}")
    print(f"  RESULTS — Wei et al. 2022 CoT  |  {cot_type.upper()}")
    print(f"  Model : {model}")
    print(f"{'='*65}")
    print(f"  Total queries : {len(df)}")
    print(f"  Accuracy (original / exact-match) : "
          f"{df['is_correct'].mean():.1%}  ({df['is_correct'].sum()}/{len(df)})")
    print(f"  Accuracy (strict  / per-dataset)  : "
          f"{df['is_correct_strict'].mean():.1%}  ({df['is_correct_strict'].sum()}/{len(df)})")
    print(f"  Total tokens  : {df['total_tokens'].sum():,}")
    print(f"  Total cost    : ${df['cost_usd'].sum():.4f}")
    print(f"  Avg latency   : {df['time_seconds'].mean():.2f}s")
    print(f"  Errors        : {df['error'].notna().sum()}")
 
    # ── Per-dataset breakdown ─────────────────────────────────
    for dataset in sorted(df["dataset"].unique()):
        ddf = df[df["dataset"] == dataset].copy()
 
        print(f"\n{'─'*65}")
        print(f"  DATASET: {dataset}  (n={len(ddf)})")
        print(f"  Accuracy (exact-match) : {ddf['is_correct'].mean():.1%}")
        print(f"  Accuracy (strict)      : {ddf['is_correct_strict'].mean():.1%}")
        print(f"  Accuracy delta         : "
              f"{(ddf['is_correct_strict'].mean() - ddf['is_correct'].mean())*100:+.1f}pp  "
              f"← gap caused by bad evaluator")
 
        # ── Sub-breakdown by level (GAIA/MATH) or subject (MMLU-PRO) ──
        group_col = "subject" if dataset == "MMLU-PRO" else "level"
        if group_col in ddf.columns and ddf[group_col].notna().any():
            grp = (
                ddf.groupby(group_col, dropna=False)
                .agg(
                    correct        =("is_correct_strict", "sum"),
                    total          =("is_correct_strict", "count"),
                    acc            =("is_correct_strict", "mean"),
                    acc_exact      =("is_correct", "mean"),
                    avg_tokens     =("total_tokens", "mean"),
                    avg_cost       =("cost_usd", "mean"),
                )
                .sort_values("acc", ascending=False)
                .reset_index()
            )
 
            header = f"  {group_col:<22} {'Correct':<9} {'Total':<7} {'Acc(strict)':<13} {'Acc(exact)':<12} {'AvgTok'}"
            print(f"\n{header}")
            print(f"  {'-'*21} {'-'*8} {'-'*6} {'-'*12} {'-'*11} {'-'*7}")
            for _, row in grp.iterrows():
                print(
                    f"  {str(row[group_col]):<22} "
                    f"{int(row['correct']):<9} "
                    f"{int(row['total']):<7} "
                    f"{row['acc']:<13.1%} "
                    f"{row['acc_exact']:<12.1%} "
                    f"{row['avg_tokens']:.0f}"
                )
 
        # ── Error Analysis ────────────────────────────────────
        errors = ddf[ddf["error"].notna()]
        if not errors.empty:
            print(f"\n  ⚠  {len(errors)} API errors in {dataset}:")
            for _, row in errors.iterrows():
                print(f"     [{row['query_id']}] {row['error'][:80]}")
 
        # ── Sample failures (up to 3) ─────────────────────────
        failures = ddf[~ddf["is_correct_strict"]].head(3)
        if not failures.empty:
            print(f"\n  Sample failures ({dataset}):")
            for _, row in failures.iterrows():
                print(f"    Q  : {str(row['question'])[:80]}...")
                print(f"    GT : {row['ground_truth']}")
                print(f"    Pred: {row['predicted']}")
                print()
 
    return df

In [185]:
all_queries = load_all()

[GAIA]      Loaded 165 rows | Columns: ['id', 'query', 'answer', 'level', 'annotator_steps', 'annotator_tools', 'file_name', 'steps_num', 'tool_num', 'time_taken']
[MMLU-PRO]  Loaded 12032 rows | Columns: ['id', 'query', 'answer', 'answer_index', 'options', 'category', 'cot_content', 'src', 'cot_length']

Total: 12197 queries across 3 datasets



In [186]:
all_queries[168]

Query(id='73', dataset='MMLU-PRO', question="_______ locate morality beyond the sphere of rationality in an emotional 'moral impulse' towards others.", ground_truth='C', level='business', options=array(['Ethical egoism', 'Ethics of duty', 'Postmodern ethics',
       'Consequentialist ethics', 'Utilitarian ethics',
       'Deontological ethics', 'Virtue ethics', 'Ethics of care',
       'Ethics of rights', 'Relativist ethics'], dtype=object), subject='business', metadata={})

In [187]:
COT_TYPE = "few_shot"    # swap to "zero_shot" for Kojima et al. style


In [188]:
for dataset_name in ["GAIA","MMLU-PRO"]:
    queries = [q for q in all_queries if q.dataset == dataset_name]
    if not queries:
        print(f"[SKIP] No queries found for {dataset_name}")
        continue

In [189]:
results_cot = run_cot_experiment(queries, cot_type=COT_TYPE)


  Wei et al. 2022 CoT — FEW_SHOT
  Model  : gpt-4o-mini
  Queries: 12032

[1/12032] MMLU-PRO | Level: business | 70
   ❓ Q  : Typical advertising regulatory bodies suggest, for example that adverts must not: encourage ________...
   ✅ GT : I
   🧠 CoT: To determine the correct option, we need to analyze the context of advertising regulations. Regulatory bodies typically...
   🤖 Ans: I
   ✓  tokens=1284  cost=$0.00026  time=3.615s
-----------------------------------------------------------------
[2/12032] MMLU-PRO | Level: business | 71
   ❓ Q  : Managers are entrusted to run the company in the best interest of ________. Specifically, they have ...
   ✅ GT : F
   🧠 CoT: Managers are entrusted to run the company in the best interest of shareholders. They have a duty to act for the benefit...
   🤖 Ans: F
   ✓  tokens=1184  cost=$0.00020  time=1.214s
-----------------------------------------------------------------
[3/12032] MMLU-PRO | Level: business | 72
   ❓ Q  : There are two main issu

KeyboardInterrupt: 

In [ ]:
evaluate(results_cot)


  RESULTS — Wei et al. 2022 CoT  |  FEW_SHOT
  Model : gpt-4o-mini
  Total queries : 10
  Accuracy (original / exact-match) : 80.0%  (8/10)
  Accuracy (strict  / per-dataset)  : 80.0%  (8/10)
  Total tokens  : 11,256
  Total cost    : $0.0018
  Avg latency   : 1.08s
  Errors        : 0

─────────────────────────────────────────────────────────────────
  DATASET: MMLU-PRO  (n=10)
  Accuracy (exact-match) : 80.0%
  Accuracy (strict)      : 80.0%
  Accuracy delta         : +0.0pp  ← gap caused by bad evaluator

  subject                Correct   Total   Acc(strict)   Acc(exact)   AvgTok
  --------------------- -------- ------ ------------ ----------- -------
  business               8         10      80.0%         80.0%        1126

  Sample failures (MMLU-PRO):
    Q  : There are two main issues associated with _____ sizing. _______ is a key issue a...
    GT : J
    Pred: D

    Q  : _______ locate morality beyond the sphere of rationality in an emotional 'moral ...
    GT : C
    Pred

,query_id,dataset,question,ground_truth,full_reasoning,predicted,is_correct,input_tokens,output_tokens,total_tokens,cost_usd,time_seconds,model,cot_type,level,subject,error,is_correct_strict
0,70,MMLU-PRO,"Typical advertising regulatory bodies suggest,...",I,The answer is I.,I,True,1122,5,1127,0.000171,0.946,gpt-4o-mini,few_shot,business,business,None,True
1,71,MMLU-PRO,Managers are entrusted to run the company in t...,F,Managers are entrusted to run the company in t...,F,True,1139,42,1181,0.000196,1.401,gpt-4o-mini,few_shot,business,business,None,True
2,72,MMLU-PRO,There are two main issues associated with ____...,J,The answer is D.,D,False,1163,5,1168,0.000177,0.723,gpt-4o-mini,few_shot,business,business,None,False
3,73,MMLU-PRO,_______ locate morality beyond the sphere of r...,C,The answer is H.,H,False,1058,5,1063,0.000162,0.538,gpt-4o-mini,few_shot,business,business,None,False
4,74,MMLU-PRO,Some of key differences between Islamic finan...,G,The answer is G.,G,True,1157,5,1162,0.000177,0.550,gpt-4o-mini,few_shot,business,business,None,True
5,75,MMLU-PRO,Which of the following are the three broad gr...,A,The three broad groups of organizational chara...,A,True,1106,34,1140,0.000186,1.369,gpt-4o-mini,few_shot,business,business,None,True
6,76,MMLU-PRO,Pine and Gilmore (1999) derive four distinct ...,D,The dimensions derived by Pine and Gilmore are...,D,True,1080,20,1100,0.000174,0.832,gpt-4o-mini,few_shot,business,business,None,True
7,77,MMLU-PRO,Which type of research methods are designed t...,J,The type of research methods designed to elici...,J,True,1047,47,1094,0.000185,1.661,gpt-4o-mini,few_shot,business,business,None,True
8,78,MMLU-PRO,Where the price is set low relative to the com...,E,The strategy of setting the price low relative...,E,True,1053,26,1079,0.000174,1.181,gpt-4o-mini,few_shot,business,business,None,True
9,79,MMLU-PRO,"Once a train pulls out of a station, or an aer...",F,The concept described refers to the fact that ...,F,True,1076,66,1142,0.000201,1.564,gpt-4o-mini,few_shot,business,business,None,True


In [165]:
def save_results_cot(results: list[Result]):
    if not results:
        raise ValueError("No results to save")

    model_tag = LLM_CONFIG["model"].replace("/", "-")
    dataset_names = sorted({r.dataset for r in results})
    dataset_slug = "_".join(
        name.lower().replace(" ", "_").replace("-", "_")
        for name in dataset_names
    )

    save_dir = os.path.join("..", "datasets", "baseline", "cot", dataset_slug)
    os.makedirs(save_dir, exist_ok=True)

    df = pd.DataFrame([asdict(r) for r in results])
    results_path = os.path.join(save_dir, f"v1_{model_tag}.parquet")
    responses_path = os.path.join(save_dir, "responses.parquet")
    metrics_path = os.path.join(save_dir, "metrics.json")

    df.to_parquet(results_path, index=False)
    df.to_parquet(responses_path, index=False)

    metrics = {
        "dataset": dataset_slug,
        "model": model_tag,
        "total_queries": len(results),
        "accuracy_exact": round(sum(r.is_correct for r in results) / len(results) * 100, 2),
        "total_tokens": int(df["total_tokens"].sum()),
        "total_cost": float(round(df["cost_usd"].sum(), 6)),
        "avg_latency": float(round(df["time_seconds"].mean(), 3)),
        "errors": int(df["error"].notna().sum()),
        "per_dataset": {},
    }

    for dataset in sorted(df["dataset"].unique()):
        ds = df[df["dataset"] == dataset]
        metrics["per_dataset"][dataset] = {
            "n": int(len(ds)),
            "accuracy_exact": round(ds["is_correct"].mean() * 100, 2),
            "total_cost": float(round(ds["cost_usd"].sum(), 6)),
            "avg_tokens": float(round(ds["total_tokens"].mean(), 1)),
        }

    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    print(f"\n✅ Saved → {results_path}")
    print(f"✅ Saved → {responses_path}")
    print(f"✅ Saved → {metrics_path}")
    print(f"   Dataset : {dataset_slug}")
    print(f"   Rows    : {len(results)}")
    print(f"   Model   : {model_tag}")
    print(f"   Correct : {sum(r.is_correct for r in results)}/{len(results)}")


In [166]:
save_results_cot(results_cot)



✅ Saved → ../datasets/baseline/cot/mmlu_pro/v1_gpt-4o-mini.parquet
✅ Saved → ../datasets/baseline/cot/mmlu_pro/responses.parquet
✅ Saved → ../datasets/baseline/cot/mmlu_pro/metrics.json
   Dataset : mmlu_pro
   Rows    : 10
   Model   : gpt-4o-mini
   Correct : 8/10
